# One-country pipeline walkthrough

Runs `backend/utils/pipeline._process_country` one step at a time, so every
intermediate is visible: the macro panel, the LLM payload, the raw article pool,
the model's subscores and per-article impacts, and the Top-3 that reach the
dashboard.

**Pick a country by editing `ISO2` in the cell under _Pick a country_, then Run All.**

Before the first run, install a kernel into the project venv (it is deliberately
not in `requirements.txt`, which is runtime-only):

```
.venv\Scripts\python.exe -m pip install ipykernel
```

Two things to know:

- **This never writes to the database.** The last step builds the payload
  `data_push.upsert_snapshot` would take and prints it, but does not call it.
  Nothing here connects to Postgres.
- **It makes real network calls**, and **Step 3 spends OpenAI credits** (one
  scoring call per run). A country with no local macro panel also hits the World
  Bank once per indicator in Step 0, which is slow.

## Setup

Resolves the repo root, loads `backend/.env`, and routes the pipeline's logging
into the notebook. `force=True` on `basicConfig` is not optional — Jupyter
installs its own root handler, so a plain `basicConfig()` is silently a no-op
and no pipeline logs appear.

In [7]:
import json
import logging
import os
import pathlib
import sys

import pandas as pd
from dotenv import load_dotenv

# Repo root = the folder holding backend/main.py, so this works whether the
# kernel's cwd is backend/ or the repo root.
PROJECT_ROOT = next(
    p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]
    if (p / "backend" / "main.py").exists()
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Same two-line load as main.py; every module reads os.getenv at call time.
load_dotenv(PROJECT_ROOT / "backend" / ".env")
load_dotenv()

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)-7s %(name)s: %(message)s",
    stream=sys.stdout,
    force=True,
)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

print(f"project root: {PROJECT_ROOT}")
for key in ("OPENAI_API_KEY", "CRAWLBASE_TOKEN", "CRAWLBASE_JS_TOKEN"):
    print(f"  {key:<20} {'set' if os.getenv(key) else 'MISSING'}")
print("  DATABASE_URL         not read - this notebook never touches Postgres")

project root: d:\Coding\ai-country-risk
  OPENAI_API_KEY       set
  CRAWLBASE_TOKEN      set
  CRAWLBASE_JS_TOKEN   set
  DATABASE_URL         not read - this notebook never touches Postgres


In [8]:
from backend.utils import constants, data_retrieval
from backend.utils.ai import client as ai_client
from backend.utils.ai import langchain_llm
from backend.utils.data_fetching import country_data_fetch
from backend.utils.news_fetching import article_enrichment, article_ranking

# Payload window and article cap, mirrored from backend/utils/pipeline.py:32-37.
SINCE_YEAR = 2015
LOOKBACK_YEARS = 10
DELTA_HORIZONS = (1, 5)
MAX_ARTICLES = 20

print(f"pipeline modules loaded - scoring model: {ai_client.MODEL_NAME}")

pipeline modules loaded - scoring model: gpt-4o-2024-08-06


## Pick a country

The table below is `constants.COUNTRY_ROSTER`. `has_panel` says whether the
macro panel is already on disk — those countries skip the slow World Bank
backfill in Step 0.

In [9]:
roster = pd.DataFrame(constants.COUNTRY_ROSTER)
roster["has_panel"] = roster["iso2"].map(
    lambda c: country_data_fetch.has_country_partition(country_data_fetch.PANEL_DIR, c)
)

print(f"{len(roster)} countries, {int(roster.has_panel.sum())} with a local macro panel")
roster.sort_values(["has_panel", "tier", "name"], ascending=[False, True, True])

48 countries, 4 with a local macro panel


,name,iso2,iso3,tier,lat,lng,has_panel
14,New Zealand,NZ,NZL,DM,-41.5000,173.0000,True
16,Portugal,PT,PRT,DM,39.7000,-8.0000,True
27,Czechia,CZ,CZE,EM,49.8200,15.4700,True
43,Taiwan,TW,TWN,EM,23.7000,120.9600,True
0,Australia,AU,AUS,DM,-24.6809,134.5300,False
1,Austria,AT,AUT,DM,47.6082,14.3738,False
2,Belgium,BE,BEL,DM,50.6003,4.7000,False
3,Canada,CA,CAN,DM,60.9215,-108.0070,False
4,Denmark,DK,DNK,DM,55.6761,10.5683,False
5,Finland,FI,FIN,DM,63.3000,25.6200,False


In [13]:
ISO2 = "US"   # <-- change country here

ENTRY = next(c for c in constants.COUNTRY_ROSTER if c["iso2"] == ISO2)
NAME, ISO3 = ENTRY["name"], ENTRY["iso3"]

print(f"{NAME}  ({ISO2}/{ISO3})  tier={ENTRY['tier']}  map=({ENTRY['lat']}, {ENTRY['lng']})")

United States  (US/USA)  tier=DM  map=(39.75, -100.5)


## Step 0 — macro panel

Each country's World Bank indicators live in a Parquet partition under
`backend/data/wb_panel_wide/country_code=XX/`. If this country has none, the
real backfill runs with the roster temporarily narrowed to this one entry — the
same trick `backend/tests/live_country_check.py` uses. Expect a few minutes and
one World Bank call per indicator.

The panel is then read back through DuckDB, exactly as the pipeline reads it.

In [14]:
if country_data_fetch.has_country_partition(country_data_fetch.PANEL_DIR, ISO2):
    print(f"panel already on disk for {ISO2}")
else:
    print(f"no panel for {ISO2} - building it (slow: one World Bank call per indicator)")
    full_roster = constants.COUNTRY_ROSTER
    constants.COUNTRY_ROSTER = [ENTRY]
    try:
        country_data_fetch.backfill_missing_panels()
    finally:
        constants.COUNTRY_ROSTER = full_roster

panel = data_retrieval.query_macro_panel(ISO2)
print(f"panel: {panel.shape[0]} years x {panel.shape[1]} columns")
panel.tail(15)

no panel for US - building it (slow: one World Bank call per indicator)
2026-07-26 14:33:03,377 INFO    backend.utils.data_fetching.country_data_fetch: Backfilling 1 missing panels → ['US']
2026-07-26 14:33:06,181 INFO    backend.utils.data_fetching.political_corruption_fetch: Fetched OWID Political Corruption Index CSV: 28708 rows.
2026-07-26 14:33:06,351 INFO    backend.utils.data_fetching.country_data_fetch: [US] Wrote panel with 237 years × 9 indicators.
panel: 26 years x 11 columns


,year,INFLATION,UNEMPLOYMENT,FDI_PCT_GDP,POL_STABILITY,RULE_OF_LAW,GINI_INDEX,GDP_PC_GROWTH,INT_PAYM_PCT_REV,POL_CORRUPTION,country_code
11,2011,3.156842,8.949,1.689112,0.624543,1.363273,41.2,0.762796,12.511076,0.054,US
12,2012,2.069337,8.069,1.540208,0.620748,1.384273,41.2,1.475706,11.664939,0.053,US
13,2013,1.464833,7.375,1.706868,0.579523,1.301608,40.9,1.348163,9.320821,0.053,US
14,2014,1.622223,6.168,1.430339,0.668123,1.341257,41.7,1.710945,9.411366,0.053,US
15,2015,0.118627,5.280,2.795482,0.665879,1.377017,41.5,2.127411,8.812689,0.053,US
16,2016,1.261583,4.869,2.522681,0.327915,1.397158,41.3,1.022666,9.911887,0.068,US
17,2017,2.130110,4.355,1.941776,0.110129,1.320034,41.4,1.750141,9.793598,0.100,US
18,2018,2.442583,3.896,1.039454,0.290496,1.172107,41.8,2.364442,12.263408,0.097,US
19,2019,1.812210,3.669,1.466965,0.054637,1.123517,41.9,2.056766,12.792787,0.093,US
20,2020,1.233584,8.055,0.641236,-0.146807,1.004381,40.0,-2.480600,10.904342,0.085,US


## Step 1 — the LLM payload

`prepare_llm_payload_pretty` compresses that panel into what the prompt actually
sees: per indicator a latest value, 1-year and 5-year percent changes, and the
last 10 observations. `_meta.generated_at` is the timestamp that would become
the snapshot's `as_of`.

`ALL_INDICATORS` is the World Bank set plus the OWID Political Corruption Index,
merged in at ingest time.

In [12]:
payload = data_retrieval.prepare_llm_payload_pretty(
    country_iso=ISO2,
    indicators=constants.ALL_INDICATORS,
    since=SINCE_YEAR,
    lookback=LOOKBACK_YEARS,
    deltas=DELTA_HORIZONS,
)

units = payload["_meta"]["units"]
indicators = pd.DataFrame([
    {
        "indicator": name,
        "unit": units.get(name, ""),
        "latest": v["latest"],
        "\u03941y": v["\u03941y"],
        "\u03945y": v["\u03945y"],
        "years": len(v["series"]),
    }
    for name, v in payload["indicators"].items()
]).set_index("indicator")

print(f"latest_year={payload['latest_year']}  generated_at={payload['_meta']['generated_at']}")
indicators

latest_year=2025  generated_at=2026-07-26T17:26Z


,unit,latest,Δ1y,Δ5y,years
indicator,,,,,
Inflation (% y/y),% y/y,2.34,-0.033,-188.808,10
Unemployment (% labour force),%,6.16,-0.052,-0.100,10
FDI inflow (% GDP),% GDP,NaN,NaN,NaN,10
Political stability (z-score),z-score,NaN,NaN,NaN,10
Rule of law (z-score),z-score,NaN,NaN,NaN,10
Income inequality (Gini),index,NaN,NaN,NaN,9
GDP per-capita growth (% y/y),% y/y,0.83,-0.250,-1.099,10
Interest payments (% revenue),% revenue,NaN,NaN,NaN,10
"Political corruption index (0–1, higher = more corrupt)",index (0–1),0.17,0.000,0.372,10


## Step 2 — news

Four Google News queries per country (broad, government, economic, security),
de-duplicated by URL and scored by the keyword heuristic in
`article_ranking.score_relevance`. Anything under 0.3 is dropped as noise —
unless that leaves fewer than 3 articles, in which case the bar is relaxed
rather than returning an empty pane.

In [ ]:
items = article_enrichment.fetch_relevant_news(NAME, max_articles=MAX_ARTICLES)

print(f"{len(items)} articles after de-dupe, scoring, and the relevance cut")
pd.DataFrame([
    {
        "relevance": it.get("relevance_score"),
        "published": it.get("published"),
        "source": it.get("source"),
        "title": it.get("title"),
    }
    for it in items
])

### Resolve and enrich

Google News links are redirect wrappers; these get unwrapped to real publisher
URLs, denylisted sources are dropped, and one GET per article recovers a
summary, body text, and thumbnail. Each survivor then gets the stable id
(`a1`, `a2`, …) the model refers back to.

In [ ]:
before_count = len(items)
items = article_enrichment.resolve_and_enrich(items, ISO2)

# Stable ids for the model to cite back (pipeline.py:128-129).
for i, it in enumerate(items, start=1):
    it["id"] = f"a{i}"

print(f"{len(items)}/{before_count} survived the source denylist")
pd.DataFrame([
    {
        "id": it["id"],
        "source": it.get("source"),
        "words": len((it.get("content") or it.get("text") or "").split()),
        "image": bool(it.get("image")),
        "title": it.get("title"),
    }
    for it in items
]).set_index("id")

## Step 3 — LLM scoring 💸

The only cell that costs money: one structured-output call rating investor risk
over the next 12 months, given the macro payload and the articles above.

It never raises. Without `OPENAI_API_KEY`, or on a network or parse failure, it
returns `score=None` and no article scores — the rest of the notebook still
runs, the tables are just empty. A country caught by the sanctions gate in
`legal_restrictions.yaml` is forced to `1.0` without calling the model at all.

In [ ]:
llm_output = langchain_llm.country_llm_score(
    country_display=NAME,
    payload=payload,
    articles=items,
)

print(f"score: {llm_output.get('score')}\n")
print(llm_output.get("bullet_summary"))

In [ ]:
display(pd.Series(llm_output.get("subscores") or {}, dtype="float64").to_frame("subscore"))

pd.DataFrame(llm_output.get("news_article_scores") or [])

## Step 4 — Top-3 selection

The model groups articles covering the same underlying event into a shared
`topic_group`. With 3 or more distinct topics, the best article of each of the
top 3 topics wins — so the dashboard shows three *stories* rather than three
write-ups of one. With fewer topics, the remainder is backfilled by impact.

The table shows every candidate, not just the winners, so you can see what lost.

In [ ]:
imp_map, topic_map = article_ranking.impact_topic_maps(llm_output)
items_by_id = {it.get("id"): it for it in items if isinstance(it, dict) and it.get("id")}
top_ids = article_ranking.select_top_ids(items_by_id, imp_map, topic_map, ISO2)

print(f"top 3: {top_ids}  (from {len(items_by_id)} candidates, "
      f"{len(set(topic_map.values()))} distinct topics)")
pd.DataFrame([
    {
        "id": aid,
        "selected": aid in top_ids,
        "impact": imp_map.get(aid),
        "topic_group": topic_map.get(aid),
        "published": it.get("published"),
        "title": it.get("title"),
    }
    for aid, it in items_by_id.items()
]).sort_values("impact", ascending=False).set_index("id")

## Step 5 — images for the Top-3

Chosen articles still missing a thumbnail get one more try through Crawlbase,
which renders JavaScript. It costs a credit per call, hence the Top-3-only
scope, and no-ops entirely without `CRAWLBASE_TOKEN`.

In [ ]:
before_images = {aid: items_by_id[aid].get("image") for aid in top_ids}
article_enrichment.enrich_top_images(top_ids, items_by_id)

pd.DataFrame([
    {"id": aid, "before": before_images[aid], "after": items_by_id[aid].get("image")}
    for aid in top_ids
]).set_index("id")

## Step 6 — the Top-3 rows

These are the rows that would be written to `risk_snapshot_article` and rendered
on the country page, followed by a rough preview of how they look there.

In [ ]:
top_articles = article_ranking.build_top_articles(top_ids, items_by_id, imp_map)
pd.DataFrame(top_articles).set_index("rank")

In [ ]:
from IPython.display import HTML

HTML("".join(
    '<div style="display:flex;gap:12px;margin:12px 0;align-items:flex-start">'
    + (f'<img src="{a["image"]}" style="width:160px;border-radius:6px">' if a["image"] else "")
    + f'<div><b>#{a["rank"]} &middot; impact {a["impact"]}</b><br>'
      f'<a href="{a["url"]}" target="_blank">{a["title"]}</a><br>'
      f'<small>{a["source"]} &middot; {a["published_at"]}</small><br>'
      f'<small>{(a["summary"] or "")[:240]}</small></div></div>'
    for a in top_articles
))

## Step 7 — the snapshot payload (not written)

The pipeline would hand this dict to `data_push.upsert_snapshot`, which writes
`country`, `indicator`, `yearly_value`, `risk_snapshot`, and
`risk_snapshot_article`.

**This notebook stops here on purpose** — no database write, so a run can never
overwrite today's real snapshot for this country. Use
`backend/tests/live_country_check.py` when you want the write plus verification
and cleanup.

In [ ]:
snapshot = {**payload, "llm_output": llm_output, "top_articles": top_articles}

# What upsert_snapshot validates before it would open a transaction.
print(f"country={snapshot['country']}  as_of<-{snapshot['_meta']['generated_at']}  "
      f"indicators={len(snapshot['indicators'])}  articles={len(snapshot['top_articles'])}")
print(json.dumps(snapshot, indent=2, default=str, ensure_ascii=False)[:3000])

## Summary

Everything the run produced, on one screen — the "did this make sense?" cell.

In [ ]:
print(f"{NAME} ({ISO2})  -  risk score {llm_output.get('score')}\n")
print(pd.Series(llm_output.get("subscores") or {}, dtype="float64").to_string(), "\n")
print(llm_output.get("bullet_summary"), "\n")
for a in top_articles:
    print(f"  #{a['rank']}  impact {a['impact']}  {a['title']}")
    print(f"      {a['source']} - {a['published_at']}")
    print(f"      {a['url']}\n")